In [1]:
import joblib

In [9]:
X_processed = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/X_processed.pkl")

y = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data/processed/y.pkl")

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, stratify=y, random_state=42
)


In [11]:
import numpy as np
import pandas as pd

print("Train target distribution:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nTest target distribution:")
print(pd.Series(y_test).value_counts(normalize=True))


Train target distribution:
target
0    0.805117
1    0.194883
Name: proportion, dtype: float64

Test target distribution:
target
0    0.805116
1    0.194884
Name: proportion, dtype: float64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [12]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
y_lr_pred = lr.predict(X_test)
y_lr_prob = lr.predict_proba(X_test)[:, 1]

print("Logistic ROC-AUC:", roc_auc_score(y_test, y_lr_prob))
print(classification_report(y_test, y_lr_pred))

Logistic ROC-AUC: 0.7578685268592368
              precision    recall  f1-score   support

           0       0.90      0.70      0.79    299557
           1       0.35      0.66      0.46     72510

    accuracy                           0.70    372067
   macro avg       0.62      0.68      0.62    372067
weighted avg       0.79      0.70      0.72    372067



In [15]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)
y_xgb_pred = xgb.predict(X_test)
y_xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("XGBoost ROC-AUC:", roc_auc_score(y_test, y_xgb_prob))
print(classification_report(y_test, y_xgb_pred))

c:\Users\siddh\miniconda3\envs\tf\lib\site-packages\xgboost\core.py:158: UserWarning: [23:40:10] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBoost ROC-AUC: 0.770601412777048
              precision    recall  f1-score   support

           0       0.90      0.72      0.80    299557
           1       0.36      0.67      0.47     72510

    accuracy                           0.71    372067
   macro avg       0.63      0.69      0.64    372067
weighted avg       0.80      0.71      0.73    372067

